Manual calculation of derivation

In [30]:
# d/dx(x^2)
def dy_dx(x):
  return 2*x


In [31]:
dy_dx(3)

6

Using PyTorch gradient

In [32]:
import torch
x = torch.tensor(3.0, requires_grad=True) #requires_grad=True means:“I want to compute derivative w.r.t. this variable”
                                          #So now PyTorch will track all operations on x

In [33]:
y = x**2 # PyTorch remembers how y depends on x (relationship)

In [34]:
x

tensor(3., requires_grad=True)

In [35]:
y

tensor(9., grad_fn=<PowBackward0>)

In [36]:
y.backward()  #Compute the derivative of y with respect to x”

# Mathematically:
#             dy/dx = 2x

In [37]:
x.grad

# dy/dx = 2x = 2×3 =6

tensor(6.)

In [38]:
# another (manual)- d/dx (sin(x**2)) = 2*cos(x**2)
import math
def dz_dx(x):
  return 2*x*torch.math.cos(x**2)

In [39]:
dz_dx(4)

-7.661275842587077

In [40]:
# pytorch
x = torch.tensor(4.0, requires_grad=True)

In [41]:
y = x**2

In [42]:
z = torch.sin(y)

In [43]:
x

tensor(4., requires_grad=True)

In [44]:
y

tensor(16., grad_fn=<PowBackward0>)

In [45]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [46]:
z.backward()

In [47]:
x.grad

tensor(-7.6613)

In [48]:
y.grad

/tmp/ipykernel_290/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  y.grad


gradient descent Manually

In [49]:
import torch

#input
x = torch.tensor(6.7) # input feature
y = torch.tensor(0.0) # ground truth

w = torch.tensor(1.0) # Weight
b = torch.tensor(0.0) # Bias

In [52]:
# BCE (loss function)

def binary_cross_entropy_loss(prediction, target):

  # Small value to avoid numerical problems
  # Why? Because log(0) is undefined (-infinity)
  # So we never want prediction to become exactly 0 or exactly 1
  epsilon = 1e-8

  # Clamp means: force values to stay inside a safe range
  prediction = torch.clamp(prediction, epsilon, 1-epsilon)

  # BCE formula
  loss = -(target*torch.log(prediction) + (1-target)* torch.log(1-prediction))

  return loss

In [53]:
# Forward pass
z = w * x + b
y_pred = torch.sigmoid(z)

# compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [54]:
loss

tensor(6.7012)

Derivative

In [55]:

# 1. dL/d(y_pred):
dloss_dy_pred = (y_pred - y)/(y_pred * (1-y_pred))

# 2. dy_pred/dz:
dy_pred_dz  = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db:
dz_dw = x
dz_db = 1

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db


In [56]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


Using Autograd

In [61]:
x = torch.tensor(6.7) # input feature
y = torch.tensor(0.0) # ground truth

w = torch.tensor(1.0, requires_grad=True) # Weight
b = torch.tensor(0.0, requires_grad=True) # Bias

In [62]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [63]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [64]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [65]:
loss.backward()

In [66]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Using Multiple inputs

In [67]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [68]:
x

tensor([1., 2., 3.], requires_grad=True)

In [69]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [70]:
y.backward()

In [71]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

Clearing grad

In [114]:
a = torch.tensor(2.0, requires_grad=True)

In [115]:
a

tensor(2., requires_grad=True)

In [116]:
b  = a**2
b

tensor(4., grad_fn=<PowBackward0>)

In [117]:
b.backward()

In [118]:
a.grad

tensor(4.)

option 1

In [119]:

a.grad.zero_()

tensor(0.)

In [120]:
# Option 2
a.requires_grad_(False)

tensor(2.)

In [121]:
a

tensor(2.)

In [122]:
b

tensor(4., grad_fn=<PowBackward0>)

In [146]:
b.backward() #-> gives error

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

Option-3 detach

In [125]:
a = torch.tensor(2.0, requires_grad=True)

In [126]:
a

tensor(2., requires_grad=True)

In [129]:
c = a.detach()

In [130]:
c

tensor(2.)

In [131]:
a

tensor(2., requires_grad=True)

In [135]:
b = a**2
b

tensor(4., grad_fn=<PowBackward0>)

In [136]:
b1 = c**2
b1

tensor(4.)

In [137]:
b.backward()

In [139]:
b1.backward() #-> gives error, backward not possible

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

option-4 (torch_no_grad)

In [141]:
a = torch.tensor(2.0, requires_grad=True)

In [142]:
a

tensor(2., requires_grad=True)

In [143]:
with torch.no_grad():
  b = a**2

In [148]:
b.backward() #-> gives error, backward not possible

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn